# DOT Traffic Analysis Pipeline
This notebook breaks down the pipeline step-by-step so you can see exactly how the data transforms at each stage.

In [1]:
import pandas as pd
import numpy as np

# --- 1. INGEST DATA ---
print("Loading data sources...")
df_trips = pd.read_parquet('data/yellow_tripdata_2026-04.parquet')
df_zones = pd.read_csv('data/taxi_zone_lookup.csv')

print(f"Loaded {len(df_trips):,} trips and {len(df_zones)} zones.")
df_trips.head(3)

Loading data sources...
Loaded 3,831,240 trips and 265 zones.


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2026-04-01 00:40:05,2026-04-01 00:52:44,1.0,2.80,1.0,N,237,68,1,15.6,4.25,0.5,4.25,0.00,1.0,25.60,2.5,0.0,0.75
1,2,2026-04-01 00:09:19,2026-04-01 00:21:29,1.0,7.37,1.0,N,138,75,1,28.2,6.00,0.5,9.03,7.46,1.0,54.19,0.0,2.0,0.00
2,2,2026-04-01 00:15:29,2026-04-01 00:34:14,1.0,7.66,1.0,N,138,112,1,31.7,6.00,0.5,5.00,0.00,1.0,46.20,0.0,2.0,0.00


### Step 2: Validate & Clean Data
Here we calculate duration, filter out impossibilities (like 0 distance, negative time, teleportation speeds).

In [2]:
# --- 2. VALIDATE & CLEAN DATA ---
initial_count = len(df_trips)

# Calculate duration in minutes
df_trips['trip_duration_minutes'] = (df_trips['tpep_dropoff_datetime'] - df_trips['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# Validation Rules
df_clean = df_trips[
    (df_trips['trip_distance'] > 0) & 
    (df_trips['trip_duration_minutes'] > 0) & 
    (df_trips['trip_duration_minutes'] < 300)
].copy()

# Calculate speed
df_clean['speed_mph'] = df_clean['trip_distance'] / (df_clean['trip_duration_minutes'] / 60.0)

# Drop impossible speeds
df_clean = df_clean[df_clean['speed_mph'] <= 80]

dropped = initial_count - len(df_clean)
print(f"Dropped {dropped:,} invalid or outlier rows.")

df_clean[['trip_distance', 'trip_duration_minutes', 'speed_mph']].describe()

Dropped 144,220 invalid or outlier rows.


,trip_distance,trip_duration_minutes,speed_mph
count,3.687020e+06,3.687020e+06,3.687020e+06
mean,3.521264e+00,1.807273e+01,1.067405e+01
std,4.321872e+00,1.481850e+01,6.310787e+00
min,1.000000e-02,1.666667e-02,5.693500e-03
25%,1.090000e+00,8.600000e+00,6.705882e+00
50%,1.900000e+00,1.416667e+01,9.067669e+00
75%,3.900000e+00,2.256667e+01,1.260517e+01
max,2.282600e+02,2.993833e+02,8.000000e+01


### Step 3: Model Workflow & Metrics
Now we map the IDs to real zone names using the CSV.

In [3]:
# --- 3. MODEL WORKFLOW & METRICS ---
# Map Pickup Locations
df_clean = df_clean.merge(df_zones[['LocationID', 'Zone']], left_on='PULocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'Zone': 'Pickup_Zone'})

# Map Dropoff Locations
df_clean = df_clean.merge(df_zones[['LocationID', 'Zone']], left_on='DOLocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'Zone': 'Dropoff_Zone'})

# Create a "Route" column
df_clean['Route'] = df_clean['Pickup_Zone'] + " to " + df_clean['Dropoff_Zone']

df_clean[['Pickup_Zone', 'Dropoff_Zone', 'Route']].head()

,Pickup_Zone,Dropoff_Zone,Route
0,Upper East Side South,East Chelsea,Upper East Side South to East Chelsea
1,LaGuardia Airport,East Harlem South,LaGuardia Airport to East Harlem South
2,LaGuardia Airport,Greenpoint,LaGuardia Airport to Greenpoint
3,LaGuardia Airport,Yorkville East,LaGuardia Airport to Yorkville East
4,Times Sq/Theatre District,Union Sq,Times Sq/Theatre District to Union Sq


### Step 4: Final Output (The 10 Slowest Routes)

In [5]:
# --- 4. OUTPUT ---
route_stats = df_clean.groupby('Route').agg(
    average_speed_mph=('speed_mph', 'mean'),
    total_trips=('speed_mph', 'count')
).reset_index()

# Filter to routes with at least 100 trips
route_stats = route_stats[route_stats['total_trips'] > 100]

# Sort by slowest routes
slowest_routes = route_stats.sort_values(by='average_speed_mph', ascending=True).head(10)

print("✅ Pipeline complete! Top 10 slowest routes:")
display(slowest_routes)

# Save to CSV
slowest_routes.to_csv('slowest_routes_report.csv', index=False)

✅ Pipeline complete! Top 10 slowest routes:


,Route,average_speed_mph,total_trips
23829,Midtown East to Times Sq/Theatre District,4.496230,3458
23593,Midtown Center to Times Sq/Theatre District,4.568104,4903
33936,Times Sq/Theatre District to Times Sq/Theatre ...,4.672677,3402
23763,Midtown East to Midtown Center,4.699963,3702
23478,Midtown Center to Garment District,4.866730,2640
27451,Penn Station/Madison Sq West to Garment District,4.908304,1575
14682,Garment District to Garment District,4.992634,736
34615,UN/Turtle Bay South to Times Sq/Theatre District,5.019477,1100
33818,Times Sq/Theatre District to Garment District,5.109556,2408
23530,Midtown Center to Midtown Center,5.110237,6158
